# Mistral AI Search Engine

When using models, it's crucial to **acknowledge** that their knowledge can be outdated and limited to their training data. For this reason, allowing the model **to access the web** is an important step in creating reliable, knowledgeable agents that can answer questions about daily and recent events, or provide rare and specialized knowledge as ground truth.

## Agents & Conversations with Web Search
We provide a web search tool out of the box, which can be used via our **Agents and Conversation API**. This API includes a list of built-in tools you can leverage, such as `websearch` and `web_search_premium`:
- `websearch`: Provides access to a **search** engine to navigate the web.
- `web_search_premium`: Provides access to a search engine as well as reliable news sources, including verified content from news providers.

### Setup
Let's set up our SDK and create an API key **[here](https://console.mistral.ai/api-keys)**.

---

In [1]:
!pip install mistralai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 854.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.3 MB/s eta 0:00:00


In [ ]:
from mistralai.client import Mistral
import getpass
import os

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

client = Mistral(os.environ["MISTRAL_API_KEY"])

With our client ready, we can now create an Agent with web search access.
When creating an Agent, you can provide a **name**, **model**, **description**, **system prompt instructions**, and a **set of tools** to use.

For this demo, we will only provide **`web_search`** access.

In [4]:
websearch_agent = client.beta.agents.create(
    model="mistral-medium-latest",
    description="Agent able to search information over the web, such as news, weather, sport results...",
    name="Websearch Agent",
    instructions="You have the ability to perform web searches with `web_search` to find up-to-date information.",
    tools=[{"type": "web_search_premium"}],
    completion_args={
        "temperature": 0.3,
        "top_p": 0.95,
    }
)
websearch_agent

Agent(model='mistral-medium-latest', name='Websearch Agent', id='ag_01a0b4a1149b7608a39df4e1d7d018b8', version=0, versions=[], created_at=datetime.datetime(2026, 9, 18, 13, 7, 30, 616139, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 9, 18, 13, 7, 30, 616143, tzinfo=TzInfo(0)), deployment_chat=False, source='api', instructions='You have the ability to perform web searches with `web_search` to find up-to-date information.', tools=[WebSearchPremiumTool(tool_configuration=None, type='web_search_premium')], completion_args=CompletionArgs(stop=None, presence_penalty=None, frequency_penalty=None, temperature=0.3, top_p=0.95, max_tokens=None, random_seed=None, prediction=None, response_format=None, tool_choice='auto', reasoning_effort=None), guardrails=[], description='Agent able to search information over the web, such as news, weather, sport results...', handoffs=None, metadata=None, object='agent', owner_id='a3499508-b2e7-440b-927b-904a41477812', version_message=None)

Agent created, we can start a conversation at any moment.

In [5]:
agent_response = client.beta.conversations.start(
    agent_id=websearch_agent.id,
    inputs="What happend during apple WWDC 2024?"
)
agent_response

ConversationResponse(conversation_id='conv_01a0b4a116757183af37549e875e7815', outputs=[ToolExecutionEntry(name='web_search', arguments='{"query": "Apple WWDC 2024 announcements and highlights"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 9, 18, 13, 7, 32, 422369, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 9, 18, 13, 7, 34, 103437, tzinfo=TzInfo(0)), agent_id='ag_01a0b4a1149b7608a39df4e1d7d018b8', model='mistral-medium-latest', id='tool_exec_01a0b4a11bc671d6960654c343122bed', info={'result': '{"mvUEGtC5": {"url": "https://www.apple.com/newsroom/2024/06/wwdc24-highlights/", "title": "WWDC24 Highlights - Apple", "description": "WWDC24 kicks off at Apple Park. Today Apple kicked off its 2024 Worldwide Developers Conference, revealing groundbreaking new technologies and features during a keynote that was live-streamed from Apple Park to millions around the world.", "snippets": ["Tim Cook greets the audience ahead of the keynote event at WWDC24.",

**Note:** It is also possible to leverage websearch without an Agent by providing a model directly without an agent id:

In [6]:
response = client.beta.conversations.start(
    model="mistral-medium-latest",
    instructions="You have the ability to perform web searches with `web_search` to find up-to-date information.",
    inputs="What happend during apple WWDC 2024?",
    tools=[{"type": "web_search_premium"}],
    # store=False
)
response

ConversationResponse(conversation_id='conv_01a0b4a1473971ac88327a931fc7c848', outputs=[ToolExecutionEntry(name='web_search_premium', arguments='{"query": "Apple WWDC 2024 announcements and highlights"}', object='entry', type='tool.execution', created_at=datetime.datetime(2026, 9, 18, 13, 7, 44, 956285, tzinfo=TzInfo(0)), completed_at=datetime.datetime(2026, 9, 18, 13, 7, 46, 875251, tzinfo=TzInfo(0)), agent_id=Unset(), model='mistral-medium-latest', id='tool_exec_01a0b4a14cbc74f085c0837f1bc47543', info={'result': '{"DOEce7MA": {"url": "https://www.apple.com/newsroom/2024/06/wwdc24-highlights/", "title": "WWDC24 Highlights - Apple", "description": "WWDC24 kicks off at Apple Park. Today Apple kicked off its 2024 Worldwide Developers Conference, revealing groundbreaking new technologies and features during a keynote that was live-streamed from Apple Park to millions around the world.", "snippets": ["Tim Cook greets the audience ahead of the keynote event at WWDC24.", "An attendee poses at

The outputs of our Agents & Conversations API includes the models reply and tool calls, let's clean and format using the references.

In [7]:
# @title Format Response
def format_response_with_references(response):
    text_parts = []
    references = []

    for output in response.outputs:
        if output.type == "message.output":
            if isinstance(output, str):
                return output
            for content in output.content:
                if content.type == "text":
                    text_parts.append(content.text)
                elif content.type == "tool_reference":
                    ref = {
                        "title": getattr(content, "title", "No Title"),
                        "url": getattr(content, "url", "No URL"),
                    }
                    # Add the reference to the list if it's not already there
                    if ref not in references:
                        references.append(ref)
                    # Get the index of the reference (now guaranteed to exist)
                    ref_index = references.index(ref) + 1
                    text_parts.append(f"[{ref_index}]")

    # Combine all text parts
    full_text = " ".join(text_parts)

    # Build the references list
    reference_list = []
    for i, ref in enumerate(references, start=1):
        reference_list.append(f"[{i}] {ref['title']} - {ref['url']}")

    # Append references to the text
    formatted_text = full_text + "\n\nReferences:\n" + "\n".join(reference_list)

    return formatted_text

# Example usage:
formatted_response = format_response_with_references(agent_response)
print(formatted_response)

Apple’s WWDC 2024 was packed with major announcements, with a strong focus on artificial intelligence and updates across all its operating systems. Here are the key highlights:

- **Apple Intelligence**: Apple introduced its new personal intelligence system, called Apple Intelligence, which integrates powerful generative AI models directly into iPhone, iPad, and Mac. This system is designed to enhance many core apps and features with AI capabilities, including Siri, which received a significant upgrade and will now be able to interact with ChatGPT for more advanced queries [1] [2] [3] .

- **iOS 18**: The latest version of iOS brings major customization options, especially for the home screen, and introduces new features in Messages, Mail, Safari, Photos, Wallet, and Notes. Safari, for example, now has AI-powered highlights to extract useful information from webpages [1] [4] [5] .

- **macOS Sequoia**: The new macOS version includes features like iPhone mirroring on Mac, allowing users

### Websearch Conversations
One of the advantages of our Agents & Conversations API compred to Chat Completions is the ability to manage multiturn and conversations out of the box in our cloud without the need of handling them locally.

When creating a conversation, by default, they are stored and you can continue the conversation at any moment. To disable this feature use `store=False`.

In [8]:
agent_new_response = client.beta.conversations.append(
    conversation_id=agent_response.conversation_id,
    inputs="Translate to French."
)

You can easily create a chat interface leveraging all these features like so:

In [9]:
conversation_id = None
while True:
    inp = input("User > ")
    if inp == "quit":
        break
    if conversation_id:
        agent_response = client.beta.conversations.start(
            agent_id=websearch_agent.id,
            inputs=inp
        )
        conversation_id = agent_response.conversation_id
    else:
        agent_response = client.beta.conversations.append(
            conversation_id=agent_response.conversation_id,
            inputs=inp
        )
    print("Agent >", format_response_with_references(agent_response))

User > What's the latest mistral ai blog post about?
Agent > The latest Mistral AI blog post, published on August 11, 2026, is titled **"In-region inference, open models, and new European infrastructure for sovereign AI."** The post discusses Mistral’s advancements in AI sovereignty, offering enterprises and countries control over AI models, infrastructure, and compute capacity to ensure regional compliance and reliability. It also announces the expansion of open model access, the introduction of regional endpoints and priority tiers, and the formation of a coalition to secure long-term European AI compute capacity [1] .

References:
[1] In-region inference, open models, and new European infrastructure for sovereign AI. - https://mistral.ai/news/regional-inference-open-models-new-compute/
User > Translate to french
Agent > Le dernier article de blog de Mistral AI, publié le 11 août 2026, s'intitule **« Inférence en région, modèles ouverts et nouvelle infrastructure européenne pour une 